In [36]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
import numpy as np
from itertools import product
from statsmodels.tsa.arima.model import ARIMA
import warnings
from joblib import Parallel, delayed
from tqdm import tqdm

In [37]:
def load_data():
    df = pd.read_csv("../data/data_preprocessed/processed_data.csv")
    df = df.set_index(["Country", "Year"])
    return df

In [38]:
df = load_data()

In [39]:
df

Capacity  Economic  Ecosystems  Exposure      Food  Governance  \
Country Year                                                                   
AFG     1995  0.893281  0.496497    0.515884  0.480512  0.665745    0.128090   
        1996  0.893892  0.496497    0.516995  0.480512  0.665583    0.128090   
        1997  0.894459  0.496497    0.518030  0.480512  0.665420    0.141392   
        1998  0.890161  0.496497    0.518156  0.480512  0.657103    0.154694   
        1999  0.885896  0.496497    0.518156  0.480512  0.648959    0.143689   
...                ...       ...         ...       ...       ...         ...   
ZWE     2019  0.597433  0.250663    0.515729  0.516912  0.544129    0.128405   
        2020  0.591558  0.260265    0.515480  0.516912  0.571499    0.124162   
        2021  0.596365  0.260265    0.515921  0.516912  0.560939    0.131438   
        2022  0.583779  0.260265    0.513962  0.516912  0.557942    0.135388   
        2023  0.573759  0.260265    0.513962  0.516912  0.559073    0.135101   

               Habitat    Health  Infrastructure  Readiness  Sensitivity  \
Country Year                                                               
AFG     1995  0.602933  0.748302        0.355586   0.308182     0.419587   
        1996  0.604235  0.748302        0.354484   0.308181     0.419307   
        1997  0.605583  0.748302        0.353925   0.312614     0.419055   
        1998  0.606909  0.748302        0.353710   0.317047     0.418794   
        1999  0.608137  0.748302        0.352292   0.313378     0.419433   
...                ...       ...             ...        ...          ...   
ZWE     2019  0.571407  0.699729        0.320293   0.170180     0.419507   
        2020  0.564062  0.699319        0.305472   0.172504     0.432905   
        2021  0.556834  0.700294        0.306890   0.177080     0.420585   
        2022  0.549853  0.699798        0.299974   0.178652     0.424039   
        2023  0.548898  0.699798        0.275926   0.178869     0.424127   

                Social  Vulnerability     Water  
Country Year                                     
AFG     1995  0.299958       0.612511  0.529692  
        1996  0.299954       0.612679  0.528281  
        1997  0.299952       0.612837  0.526853  
        1998  0.299949       0.611179  0.525424  
        1999  0.299947       0.609828  0.525585  
...                ...            ...       ...  
ZWE     2019  0.131472       0.505822  0.383647  
        2020  0.133086       0.507918  0.391679  
        2021  0.139537       0.505453  0.391842  
        2022  0.140304       0.502217  0.391776  
        2023  0.141240       0.498239  0.391776  

[5394 rows x 14 columns]

In [40]:
indicators = df.columns
countries = df.index.get_level_values("Country").unique()

In [41]:
results = []

for country in countries:
    for indicator in indicators:

        series = df.loc[country, indicator].dropna()

        try:
            p_value = adfuller(series)[1]
        except ValueError:
            p_value = None

        results.append({
            "Country": country,
            "Indicator": indicator,
            "p_value": p_value
        })

adf_results = pd.DataFrame(results)

/Users/deniz/.pyenv/versions/3.10.6/envs/climate-resilience-dashboard/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:955: RuntimeWarning: divide by zero encountered in log
  llf = -nobs2*np.log(2*np.pi) - nobs2*np.log(ssr / nobs) - nobs2
/Users/deniz/.pyenv/versions/3.10.6/envs/climate-resilience-dashboard/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:955: RuntimeWarning: divide by zero encountered in log
  llf = -nobs2*np.log(2*np.pi) - nobs2*np.log(ssr / nobs) - nobs2
/Users/deniz/.pyenv/versions/3.10.6/envs/climate-resilience-dashboard/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:955: RuntimeWarning: divide by zero encountered in log
  llf = -nobs2*np.log(2*np.pi) - nobs2*np.log(ssr / nobs) - nobs2
/Users/deniz/.pyenv/versions/3.10.6/envs/climate-resilience-dashboard/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:955: RuntimeWarning: divide by zero encountered in log
  llf = -nobs2*np.log(2*np.pi)

In [42]:
adf_table = adf_results.pivot(
    index="Country",
    columns="Indicator",
    values="p_value"
)

In [43]:
(adf_table > 0.05).sum()

Indicator
Capacity          165
Economic          165
Ecosystems        115
Exposure            0
Food              157
Governance        156
Habitat           151
Health            168
Infrastructure    170
Readiness         166
Sensitivity       151
Social            162
Vulnerability     178
Water             146
dtype: int64

### p > 0.05 → fail to reject the null hypothesis → the series is likely non-stationary.

In [17]:
def fit_arima(series, p, d, q):
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model = ARIMA(series, order=(p, d, q)).fit()
        return (p, d, q), model.aic, model
    except:
        return None

results = []

for country in countries:
    for indicator in indicators:
        series = df.loc[country, indicator]

        # Fit all ARIMA combinations in parallel
        results_grid = Parallel(n_jobs=-1)(
            delayed(fit_arima)(series, p, d, q)
            for p, d, q in tqdm(
                product(range(4), range(3), range(4)),
                desc=f"{country} - {indicator}",
                total=48  # 4 × 3 × 4
            )
        )

        # Find best model
        valid_results = [r for r in results_grid if r is not None]

        if not valid_results:
            print(f"Skipping {country} - {indicator}")
            continue

        best_order, best_aic, best_model = min(valid_results, key=lambda x: x[1])

        results.append({
            "Country": country,
            "Indicator": indicator,
            "Order": best_order,
            "AIC": best_aic,
            "BIC": best_model.bic,
        })

# Convert to DataFrame if needed
import pandas as pd
df_results = pd.DataFrame(results)

ZWE - Water: 100%|██████████| 48/48 [00:00<00:00, 828.89it/s]


In [22]:
results_df = pd.DataFrame(results)

In [23]:
results_df

,Country,Indicator,Order,AIC,BIC
0,AFG,Capacity,"(3, 0, 1)",-167.254275,-159.050500
1,AFG,Economic,"(0, 1, 1)",-103.036247,-100.371838
2,AFG,Ecosystems,"(1, 0, 0)",-270.389178,-266.287291
3,AFG,Exposure,"(0, 1, 0)",-689.299058,-687.966853
4,AFG,Food,"(1, 0, 2)",-177.312727,-170.476247
...,...,...,...,...,...
2599,ZWE,Readiness,"(2, 0, 0)",-186.989915,-181.520732
2600,ZWE,Sensitivity,"(1, 0, 0)",-183.861247,-179.759360
2601,ZWE,Social,"(2, 0, 2)",-230.876638,-222.672863
2602,ZWE,Vulnerability,"(1, 0, 0)",-232.783278,-228.681391


In [12]:
results = pd.read_csv("model_results_all_sectors.csv")

In [26]:
order_df = pd.DataFrame(results.groupby(['Indicator','Order']).size())

In [31]:
order_df.head()

0
Indicator Order        
Capacity  (0, 0, 1)   4
          (0, 0, 3)   2
          (0, 1, 0)  12
          (0, 1, 1)   4
          (0, 1, 2)   4

In [32]:
order_df = order_df.reset_index()
order_df.columns = ['Indicator', 'Order', 'Count']

In [35]:
# Smallest (p,d,q) for each indicator
smallest = (
    order_df.groupby('Indicator')['Order']
    .min()
    .rename('Smallest_Order')
)

# Most frequent order for each indicator
most_frequent = (
    order_df.loc[
        order_df.groupby('Indicator')['Count'].idxmax(),
        ['Indicator', 'Order', 'Count']
    ]
    .rename(columns={
        'Order': 'Most_Frequent_Order',
        'Count': 'Frequency'
    })
    .set_index('Indicator')
)

summary = (
    smallest.to_frame()
    .join(most_frequent)
    .reset_index()
)

summary

,Indicator,Smallest_Order,Most_Frequent_Order,Frequency
0,Capacity,"(0, 0, 1)","(1, 0, 0)",42
1,Economic,"(0, 0, 0)","(1, 0, 0)",56
2,Ecosystems,"(0, 0, 0)","(2, 0, 0)",58
3,Exposure,"(0, 1, 0)","(0, 1, 0)",186
4,Food,"(0, 0, 0)","(1, 0, 0)",79
5,Governance,"(0, 0, 1)","(1, 0, 0)",60
6,Habitat,"(0, 0, 1)","(1, 0, 1)",53
7,Health,"(0, 0, 1)","(1, 0, 0)",63
8,Infrastructure,"(0, 0, 0)","(1, 0, 0)",36
9,Readiness,"(0, 0, 1)","(1, 0, 0)",52


In [52]:
summary['Smallest_Order'].unique()

array(['(0, 0, 1)', '(0, 0, 0)', '(0, 1, 0)'], dtype=object)

In [53]:
summary['Most_Frequent_Order'].unique()

array(['(1, 0, 0)', '(2, 0, 0)', '(0, 1, 0)', '(1, 0, 1)', '(2, 0, 1)'],
      dtype=object)

In [54]:
grid = [(0, 0, 1), (0, 0, 0), (0, 1, 0), (1, 0, 0), (2, 0, 0), (1, 0, 0), (2, 0, 1)]

## Running Grid Search based on results of previous run.

In [55]:
def fit_arima(series, p, d, q):
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model = ARIMA(series, order=(p, d, q)).fit()

            # Calculate MSE from residuals
            mse = np.mean(model.resid ** 2)

            return (p, d, q), model.aic, model.bic, mse, model
    except:
        return None

# Your grid search results from yesterday
grid = [(0, 0, 1), (0, 0, 0), (0, 1, 0), (1, 0, 0), (2, 0, 0), (1, 0, 0), (2, 0, 1)]

results = []

for country in countries:
    for indicator in indicators:
        series = df.loc[country, indicator]

        # Fit only the grid combinations
        results_grid = Parallel(n_jobs=-1)(
            delayed(fit_arima)(series, p, d, q)
            for p, d, q in tqdm(
                grid,
                desc=f"{country} - {indicator}",
                total=len(grid)
            )
        )

        # Find best model by AIC
        valid_results = [r for r in results_grid if r is not None]

        if not valid_results:
            print(f"Skipping {country} - {indicator}")
            continue

        best_order, best_aic, best_bic, best_mse, best_model = min(
            valid_results, key=lambda x: x[1]  # Still using AIC as criterion
        )

        results.append({
            "Country": country,
            "Indicator": indicator,
            "Order": best_order,
            "AIC": best_aic,
            "BIC": best_bic,
            "MSE": best_mse,
        })

# Convert to DataFrame
df_results = pd.DataFrame(results)

ZWE - Water: 100%|██████████| 7/7 [00:00<00:00, 17538.91it/s]


In [56]:
df_results

,Country,Indicator,Order,AIC,BIC,MSE
0,AFG,Capacity,"(0, 1, 0)",-159.674542,-158.342338,0.027691
1,AFG,Economic,"(0, 1, 0)",-101.432891,-100.100687,0.009906
2,AFG,Ecosystems,"(1, 0, 0)",-270.389178,-266.287291,0.000004
3,AFG,Exposure,"(0, 1, 0)",-689.299058,-687.966853,0.007962
4,AFG,Food,"(1, 0, 0)",-176.504170,-172.402282,0.000170
...,...,...,...,...,...,...
2599,ZWE,Readiness,"(2, 0, 0)",-186.989915,-181.520732,0.000219
2600,ZWE,Sensitivity,"(1, 0, 0)",-183.861247,-179.759360,0.000098
2601,ZWE,Social,"(1, 0, 0)",-227.030455,-222.928567,0.000039
2602,ZWE,Vulnerability,"(1, 0, 0)",-232.783278,-228.681391,0.000015
